In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src import utils

In [ ]:
from huggingface_hub import HfFolder, login

api_file = "/home/fre.gilad/source/llm-iml/HF_KEY.txt"
hf_token = utils.api_key_from_file(api_file)

HfFolder.save_token(hf_token)
login(token=hf_token)

In [ ]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

# shuffle the data
utils.set_seed(42)
data = data.sample(frac=1).reset_index(drop=True)

In [ ]:
split = int(0.5 * len(data))
ds_train = data.iloc[:split].copy()
ds_eval = data.iloc[split:].copy()

# ds_train = data.copy()
# ds_eval = data.copy()

dl_train = DF_Batcher(ds_train, batch_size=10, shuffle=True)
dl_eval = DF_Batcher(ds_eval, batch_size=25, shuffle=False)

In [ ]:
print("Train dataset size:", len(ds_train))
print("Eval dataset size:", len(ds_eval))

In [ ]:
from src.eval.harmbench_evaluator import HarmBenchEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from gserve.configs import ServeConfig, LLMConfig


evaluators = [
    HarmBenchEvaluator(
        serve_config=ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=60),
    ),
    StrongRejectEvaluator(
        serve_config=ServeConfig(gpu_ids=[2], startup_timeout=20 * 60, client_timeout=60),
        binary_thresh=0.5,
    ),
    TemplateEvaluator(),
]

In [ ]:
import torch
from notebooks.utils import print_supported_models, load_model

torch.set_float32_matmul_precision("high")

print_supported_models()

In [ ]:
model, tokenizer = load_model("meta-llama/Llama-2-7b-chat-hf")

In [ ]:
print(model)

In [ ]:
from src.univ_attacks import UnivSoftPrompt
from src.adv_model import AdvModel
from src.initialize import Initializer
from src.config import GenConfig, StopCriteria
from src.fgsm_optim import FGSM

adv_model = AdvModel(model=model, tokenizer=tokenizer, num_tokens=20)

Initializer.from_string(adv_model, "! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! !", strict=False)

gen_config = GenConfig(
    max_new_tokens=512,
    do_sample=False,
    remove_invalid_values=True,
)

optimizer = FGSM(
    adv_model.parameters(),
    lr=0.001,
)

univ_attack = UnivSoftPrompt(
    adv_model=adv_model,
    optimizer=optimizer,
    evaluators=evaluators,
    eval_freq=2,
    gen_config=gen_config,
    mixed_precision=False,
    log_dir="logs",
)

stop = StopCriteria(max_epochs=200, max_time=60 * 60)

In [ ]:
adv_model = univ_attack.fit(dl_train, dl_eval, stop_criteria=stop)
univ_attack.close()

In [ ]:
adv_model.set_embeddings(univ_attack.best_embeds)
adv_model.discretize()
univ_attack.evaluate(adv_model=adv_model, dl_eval=dl_eval, evaluators=evaluators)

In [ ]:
preds = univ_attack.predict(adv_model, dl_eval, gen_config)
dl_eval.set_column("response", preds)

for i in range(len(preds)):
    print(" == Prompt:")
    print(ds_eval.iloc[i]["prompt"])
    print(" == Target:")
    print(ds_eval.iloc[i]["target"])
    print(" == Prediction:")
    print(preds[i])
    print("\n" + "=" * 50 + "\n")